In [1]:
# %load_ext autoreload
# %autoreload 2

In [2]:
# if google colab
use_colab = True

# download packages
!pip install pypose
!pip install kornia

# mount drive with data
from google.colab import drive
drive.mount('/content/drive')

# clone repo
%cd /content/
!git clone https://github.com/MarcinJanis/SonarOdometry.git
%cd SonarOdometry



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 37.9 MB/s eta 0:00:00
Mounted at /content/drive
/content
Cloning into 'SonarOdometry'...
remote: Enumerating objects: 4836, done.
remote: Counting objects: 100% (379/379), done.
remote: Compressing objects: 100% (287/287), done.
remote: Total 4836 (delta 243), reused 161 (delta 87), pack-reused 4457 (from 3)
Receiving objects: 100% (4836/4836), 632.28 MiB | 18.84 MiB/s, done.
Resolving deltas: 100% (1826/1826), done.
Updating files: 100% (266/266), done.
/content/SonarOdometry


In [3]:
import torch
import torch.nn.functional as F
from torchvision.transforms import v2
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib import cm
import os, sys
import pandas as pd

import plotly.graph_objects as go

from box import Box
import yaml

root_dir = os.path.abspath('../..')

if root_dir not in sys.path:
    sys.path.append(root_dir)

from src.data_loader.utils import img_polar2cart
from src.data_loader.transforms import SpeckleNoise, RayArtifacts
from src.data_loader.evaluation_data_generator import DataGenerator


In [4]:
def visu_polar_and_carth(i_polars, row_titles=None, r_min=0.2, r_max=100.0, fov=135 * np.pi / 180, cmap = 'inferno'):
    if not isinstance(i_polars, list):
        i_polars = [i_polars]

    n = len(i_polars)

    if row_titles is not None:
        if not isinstance(row_titles, list):
            row_titles = [row_titles]
        if len(row_titles) < n:
            row_titles.extend([""] * (n - len(row_titles)))

    first_cart = img_polar2cart(i_polars[0], r_min, r_max, fov, out_shape=None, bg=0)
    W_pol = i_polars[0].shape[1]
    W_cart = first_cart.shape[1]

    fig, ax = plt.subplots(
        n, 2,
        figsize=(15, 5 * n),
        squeeze=False,
        gridspec_kw={
            'width_ratios': [W_pol, W_cart],
            'wspace': 0.05,
            'hspace': 0.01
        }
    )

    for idx, i_polar in enumerate(i_polars):
        i_cart = img_polar2cart(i_polar, r_min, r_max, fov, out_shape=None, bg=0)

        # Tytuł dla całego wiersza
        if row_titles and row_titles[idx]:
            ax[idx, 0].text(-0.25, 0.5, row_titles[idx], transform=ax[idx, 0].transAxes,
                            rotation=90, va='center', ha='right', fontsize=12, fontweight='bold')

        # === Układ polarny ===
        ax[idx, 0].imshow(i_polar[:, :], cmap=cmap)
        ax[idx, 0].axis('off')

        # === Układ kartezjański ===
        ax[idx, 1].imshow(i_cart[:, :], cmap=cmap)
        ax[idx, 1].axis('off')


    plt.tight_layout(w_pad=0.05)
    plt.show()



def visu_single(imgs, row_titles=None, cmap='inferno'):
    if not isinstance(imgs, list):
        imgs = [imgs]

    n = len(imgs)

    if row_titles is not None:
        if not isinstance(row_titles, list):
            row_titles = [row_titles]
        if len(row_titles) < n:
            row_titles.extend([""] * (n - len(row_titles)))

    fig, ax = plt.subplots(
        n, 1,
        figsize=(7, 5 * n),
        squeeze=False
    )

    for idx, img in enumerate(imgs):
        if row_titles and row_titles[idx]:
            ax[idx, 0].text(
                -0.15, 0.5,
                row_titles[idx],
                transform=ax[idx, 0].transAxes,
                rotation=90,
                va='center',
                ha='right',
                fontsize=12,
                fontweight='bold'
            )

        ax[idx, 0].imshow(img, cmap=cmap)
        ax[idx, 0].axis('off')

    plt.tight_layout()
    plt.show()

In [27]:
use_colab = True
if use_colab:
    root_dir = '/content/SonarOdometry'
    data_root_dir =  '/content/drive/MyDrive/Studia/SonarOdometryDataset/SonarOdometryDataset_sample/test_scenarios/seq_3'
else:
    root_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry'
    data_root_dir =  os.path.join(root_dir, 'SonarOdometryDataset/test_scenarios/seq_3')


model_config_pth = os.path.join(root_dir, 'config/model2d.yaml')
sonar_config_pth = os.path.join(root_dir, 'config/sonar.yaml')

image_num = 15

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(model_config_pth, "r") as f:
            model_config = Box(yaml.safe_load(f))

with open(sonar_config_pth, "r") as f:
            sonar_config = Box(yaml.safe_load(f))

from src.models.utils import ExtrinsicsCalib



# from src.data_loader.transforms import SonarDatasetTranforms

fls_resolution = (model_config.input.polar_height, model_config.input.polar_width)

m = 7.250794635405014
transform = v2.Compose([
    RayArtifacts(0.0, 0.03, 0, 10, num_rays=4, probability=1.0),
    v2.GaussianBlur(kernel_size=(7, 5), sigma=(1.0, 2.0)),
    SpeckleNoise(concentration=m, rate=m)])

data_generator = DataGenerator(data_root_dir, device, fls_resolution=fls_resolution, transforms = transform, calibration = None)

r_min = 0.5 #sonar_config.range.min
r_max = 100.0 #sonar_config.range.min
fov = sonar_config.fov.horizontal

In [ ]:
t, frame_noise, pose_gt, depth= data_generator.get_sample(image_num, return_visu=False, return_depth=True)
# frame_noise_visu = frame_noise.squeeze().unsqueeze(-1).detach().cpu().numpy()
frame_np_norm = frame_noise.squeeze().detach().cpu().numpy()
frame_np = np.clip(frame_np_norm * 255.0, 0, 255).astype(np.uint8)
visu_polar_and_carth(frame_np[..., np.newaxis], row_titles='Zaszumiony obraz', r_min=0.2, r_max=100.0, fov=135 * np.pi / 180, cmap = 'inferno')

# aracati_np = cv2.imread(aracati_pth, 0)
# visu_single(aracati_np[..., np.newaxis], row_titles='Zaszumiony obraz')

Filtr Medianowy

In [ ]:
frame_blured = cv2.medianBlur(frame_np, ksize=5)
aracti_blured = cv2.medianBlur(aracati_np, ksize=5)

visu_polar_and_carth([frame_np[..., np.newaxis],
      frame_blured[..., np.newaxis]],
      row_titles=['Zaszumiony obraz',
                  'Filtr medianowy'],
                  r_min=0.2, r_max=100.0, fov=135 * np.pi / 180, cmap = 'inferno')

# visu_single([aracati_np[..., np.newaxis],
#              aracti_blured[..., np.newaxis]],
#              row_titles=['Zaszumiony obraz',
#                          'Filtr medianowy'])

In [ ]:
frame_bilatelar = cv2.bilateralFilter(frame_blured, d=5, sigmaColor=50, sigmaSpace=7)
aracati_bilatelar = cv2.bilateralFilter(aracti_blured, d=5, sigmaColor=50, sigmaSpace=7)

visu_polar_and_carth([frame_np[..., np.newaxis],
      frame_bilatelar[..., np.newaxis]],
      row_titles=['Zaszumiony obraz',
                  'Filtr medianowy'],
                  r_min=0.2, r_max=100.0, fov=135 * np.pi / 180, cmap = 'inferno')


# visu_single([aracati_np[..., np.newaxis],
#              aracati_bilatelar[..., np.newaxis]],
#              row_titles=['Zaszumiony obraz',
#                          'Filtr medianowy'])

In [ ]:
clahe = cv2.createCLAHE(clipLimit=2, tileGridSize=(8, 8))
filtered = clahe.apply(frame_bilatelar)

clahe = cv2.createCLAHE(clipLimit=2, tileGridSize=(8, 8))
aracati_filtered = clahe.apply(aracati_bilatelar)

visu_polar_and_carth([frame_np[..., np.newaxis],
      filtered[..., np.newaxis]],
      row_titles=['Zaszumiony obraz',
                  'Filtr medianowy'],
                  r_min=0.2, r_max=100.0, fov=135 * np.pi / 180, cmap = 'inferno')

# visu_single([aracati_np[..., np.newaxis],
#              aracati_filtered[..., np.newaxis]],
#              row_titles=['Zaszumiony obraz',
#                          'Filtr medianowy'])

In [ ]:
# get together

visu_polar_and_carth([frame_np[..., np.newaxis],
                      frame_blured[..., np.newaxis],
                      frame_bilatelar[..., np.newaxis],
                      filtered[..., np.newaxis]],

      row_titles=['Zaszumiony obraz',
                  'Filtr medianowy',
                  'filtr bilateralny',
                  'CAHE'],
                  r_min=0.2, r_max=100.0, fov=135 * np.pi / 180, cmap = 'inferno')

In [39]:
def fls_filter(frame):
       device = frame.device
       b, c, h, w = frame.shape
       # norm torch tensor -> np array
       frame_np = frame.squeeze().detach().cpu().numpy()
       frame_uint8 = np.clip(frame_np * 255.0, 0, 255).astype(np.uint8)
       # median blur
       blured = cv2.medianBlur(frame_uint8, ksize=3)
       # bilateral filter
       bilateral = cv2.bilateralFilter(blured, d=5, sigmaColor=25, sigmaSpace=5)
       # histogram equalization - CLAHE
       clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
       filtered = clahe.apply(bilateral)
       # np array -> norm torch tensor
       filtered_float = filtered.astype(np.float32) / 255.0
       frame_torch = torch.tensor(filtered_float, device=device).unsqueeze(0)
       return frame_torch

Filtration quality evaluation

In [85]:
with torch.no_grad():
  image_num = 3313
  image_num_offset = -4

  # read images
  t1, frame1, pose_gt1, depth1 = data_generator.get_sample(image_num, return_visu=False, return_depth=True)
  t2, frame2, pose_gt2, depth2 = data_generator.get_sample(image_num + image_num_offset, return_visu=False, return_depth=True)

  # filtration
  frame1_uf = frame1.squeeze(0).squeeze(0)
  frame2_uf = frame2.squeeze(0).squeeze(0)
  frame1_f = fls_filter(frame1.squeeze(0))
  frame2_f = fls_filter(frame2.squeeze(0))

  print(frame1_f.max())
  print(frame1_uf.max())

  # sampling grid for polar -> cart transformation
  c, h, w = frame1_f.shape
  b = 1
  out_h, out_w = h, 2 * h
  y = torch.arange(out_h, device=device, dtype=torch.float32)
  x = torch.arange(out_w, device=device, dtype=torch.float32)
  y, x = torch.meshgrid(y, x, indexing='ij')
  x = x - out_w / 2.0
  y = out_h - y

  scale = (r_max - r_min) / out_h
  x_r = x * scale
  y_r = y * scale + r_min
  r = torch.sqrt(x_r**2 + y_r**2)
  theta = torch.atan2(x_r, torch.clamp(y_r, min=1e-5))

  norm_theta = theta / (fov / 2.0)
  norm_r = (r - r_min) / (r_max - r_min) * 2.0 - 1.0

  polar2cart_grid = torch.stack((norm_theta, -norm_r), dim=-1).unsqueeze(0)

  # transform frames to cart
  frame1_c_f = F.grid_sample(frame1_f.unsqueeze(0), polar2cart_grid, mode='bilinear', padding_mode='zeros', align_corners=True)
  frame2_c_f = F.grid_sample(frame2_f.unsqueeze(0), polar2cart_grid, mode='bilinear', padding_mode='zeros', align_corners=True)
  frame1_c_uf = F.grid_sample(frame1_uf.unsqueeze(0), polar2cart_grid, mode='bilinear', padding_mode='zeros', align_corners=True)
  frame2_c_uf = F.grid_sample(frame2_uf.unsqueeze(0), polar2cart_grid, mode='bilinear', padding_mode='zeros', align_corners=True)

  # creare mask
  valid_mask = (norm_theta >= -1.0) & (norm_theta <= 1.0) & (norm_r >= -1.0) & (norm_r <= 1.0)
  print(valid_mask.sum())
  mask = valid_mask.unsqueeze(0).expand(b, -1, -1).float()

  frame1_c_f *= mask
  frame2_c_f *= mask
  frame1_c_uf *= mask
  frame2_c_uf *= mask

  print(frame1_c_f.max())
  print(frame1_c_uf.max())

  print(frame1_c_f.shape)
  print(frame2_c_uf.shape)
  print(mask.shape)

tensor(0.8157, device='cuda:0')
tensor(0.7049, device='cuda:0')
tensor(675482, device='cuda:0')
tensor(0.8146, device='cuda:0')
tensor(0.6997, device='cuda:0')
torch.Size([1, 1, 768, 1536])
torch.Size([1, 1, 768, 1536])
torch.Size([1, 768, 1536])


In [15]:
from kornia.feature import LoFTR
match_points = LoFTR(pretrained='outdoor').to(device).eval()

Downloading: "http://cmp.felk.cvut.cz/~mishkdmy/models/loftr_outdoor.ckpt" to /root/.cache/torch/hub/checkpoints/loftr_outdoor.ckpt


100%|██████████| 44.2M/44.2M [00:03<00:00, 14.5MB/s]


In [41]:
def scale_px2physcial(pts_px, input_fromat = 'polar'):
        # scaling pts in carthesian frame of reference
        out_h, out_w = frame1_c_f.shape[-2], frame1_c_f.shape[-1]

        if input_fromat== 'polar':
            resolution_m_per_px = (r_max - r_min) / out_h
            x = (pts_px[:, 0] - out_w / 2.0) * resolution_m_per_px
            y = (out_h - pts_px[:, 1]) * resolution_m_per_px + r_min
        else:
            res_y = (r_max - r_min) / out_h
            physical_width = 2.0 * r_max * np.sin(fov/ 2.0)
            res_x = physical_width / out_w
            x = (pts_px[:, 0] - out_w / 2.0) * res_x
            y = (out_h - pts_px[:, 1]) * res_y + r_min

        return torch.stack([x, y], dim=1)

In [86]:
# unfiltered

with torch.inference_mode():
  matches = match_points({'image0': frame1_c_uf,
                          'mask0': mask,
                          'image1': frame2_c_uf,
                          'mask1': mask})

pts1, pts2, confidence = matches['keypoints0'], matches['keypoints1'], matches['confidence']

print(len(pts1))
# === Statistics ===

# Max possible pts pairs
b, c, h, w = frame1_c_uf.shape
max_pts_full_img = h*w / (8*8) # if img didn't have blind areas
max_pts_real = max_pts_full_img * float(torch.sum(mask.flatten())) / (h*w) # proportionaly to real img area

# Matched points
pts_matched = len(pts1)

print('Unfiltered')
print(f'Matched points: {pts_matched}/{max_pts_real} ({pts_matched/max_pts_real})')

# Confidence
print(f'Confidence: mean: {float(torch.mean(confidence))}, std: {float(torch.std(confidence))}, median: {float(torch.median(confidence))}')

# scaleing, depth copensation

theta_max = fov / 2

valid_matches = confidence > model_config.feature_matching.pts_match_thresh
pts1, pts2, confidence = pts1[valid_matches], pts2[valid_matches], confidence[valid_matches]

pts1_visu = pts1
pts2_visu = pts2

# Scale for phisical values
pts1_r, pts2_r = scale_px2physcial(pts1), scale_px2physcial(pts2)

#
valid_mask = torch.ones(len(pts1_r), dtype=torch.bool, device=device)
ray1 = torch.sqrt(pts1_r[:, 0]**2 + pts1_r[:, 1]**2)
ray2 = torch.sqrt(pts2_r[:, 0]**2 + pts2_r[:, 1]**2)

# Range masking
use_range_masking = True
max_valid_range_ratio = model_config.filtering.range_masking_max_range
if use_range_masking:

    valid_mask = valid_mask & (ray1 > depth1) & (ray2 > depth2)
    valid_mask = valid_mask & (ray1 < r_max * max_valid_range_ratio) & (ray2 < r_max * max_valid_range_ratio)

    pts1_r, pts2_r = pts1_r[valid_mask], pts2_r[valid_mask]
    pts1, pts2 = pts1[valid_mask], pts2[valid_mask]
    ray1, ray2 = ray1[valid_mask], ray2[valid_mask]
    confidence = confidence[valid_mask]

    pts1_visu = pts1_visu[valid_mask]
    pts2_visu = pts2_visu[valid_mask]

# Depth compensation
depth_compesation = True
if depth_compesation:
    scale1 = torch.sqrt(ray1**2 - depth1**2) / ray1
    scale2 = torch.sqrt(ray2**2 - depth2**2) / ray2
    pts1_r = pts1_r * scale1.unsqueeze(1)
    pts2_r = pts2_r * scale2.unsqueeze(1)

pts1_np, pts2_np = pts1_r.cpu().numpy(), pts2_r.cpu().numpy()
conf_np = confidence.cpu().numpy()

M, inlier_mask = cv2.estimateAffinePartial2D(
pts1_np, pts2_np, method=cv2.RANSAC,
ransacReprojThreshold= model_config.feature_matching.pts_match_thresh,
maxIters=3000,
confidence=0.999
)

if M is not None and inlier_mask is not None:
    inlier_mask = inlier_mask.ravel().astype(bool)
    inliers_abs = int(inlier_mask.sum())
    outliers_abs = len(pts1) - inliers_abs
    inliers_p = inliers_abs / len(pts1) if len(pts1) > 0 else 0.0
    outliers_p = outliers_abs / len(pts1) if len(pts1) > 0 else 0.0
print(f'Points matching parameters: \n inliers: {inliers_abs} ({inliers_p}) \n outliers: {outliers_abs} ({outliers_p})')

4347
Unfiltered
Matched points: 4347/10554.40625 (0.4118658972407851)
Confidence: mean: 0.49257710576057434, std: 0.20903374254703522, median: 0.45567354559898376
Points matching parameters: 
 inliers: 1578 (0.8063362289218191) 
 outliers: 379 (0.1936637710781809)


In [87]:
# unfiltered

with torch.inference_mode():
  matches = match_points({'image0': frame1_c_f,
                          'mask0': mask,
                          'image1': frame2_c_f,
                          'mask1': mask})

pts1, pts2, confidence = matches['keypoints0'], matches['keypoints1'], matches['confidence']

# === Statistics ===

# Max possible pts pairs
b, c, h, w = frame1_c_uf.shape
max_pts_full_img = h*w / (8*8) # if img didn't have blind areas
max_pts_real = max_pts_full_img * float(torch.sum(mask.flatten())) / (h*w) # proportionaly to real img area

# Matched points
pts_matched = len(pts1)

print('Filtered')
print(f'Matched points: {pts_matched}/{max_pts_real} ({pts_matched/max_pts_real})')

# Confidence
print(f'Confidence: mean: {float(torch.mean(confidence))}, std: {float(torch.std(confidence))}, median: {float(torch.median(confidence))}')

# scaleing, depth copensation

theta_max = fov / 2

valid_matches = confidence > model_config.feature_matching.pts_match_thresh
pts1, pts2, confidence = pts1[valid_matches], pts2[valid_matches], confidence[valid_matches]

pts1_visu = pts1
pts2_visu = pts2

# Scale for phisical values
pts1_r, pts2_r = scale_px2physcial(pts1), scale_px2physcial(pts2)

#
valid_mask = torch.ones(len(pts1_r), dtype=torch.bool, device=device)
ray1 = torch.sqrt(pts1_r[:, 0]**2 + pts1_r[:, 1]**2)
ray2 = torch.sqrt(pts2_r[:, 0]**2 + pts2_r[:, 1]**2)

# Range masking
use_range_masking = True
max_valid_range_ratio = model_config.filtering.range_masking_max_range
if use_range_masking:

    valid_mask = valid_mask & (ray1 > depth1) & (ray2 > depth2)
    valid_mask = valid_mask & (ray1 < r_max * max_valid_range_ratio) & (ray2 < r_max * max_valid_range_ratio)

    pts1_r, pts2_r = pts1_r[valid_mask], pts2_r[valid_mask]
    pts1, pts2 = pts1[valid_mask], pts2[valid_mask]
    ray1, ray2 = ray1[valid_mask], ray2[valid_mask]
    confidence = confidence[valid_mask]

    pts1_visu = pts1_visu[valid_mask]
    pts2_visu = pts2_visu[valid_mask]

# Depth compensation
depth_compesation = True
if depth_compesation:
    scale1 = torch.sqrt(ray1**2 - depth1**2) / ray1
    scale2 = torch.sqrt(ray2**2 - depth2**2) / ray2
    pts1_r = pts1_r * scale1.unsqueeze(1)
    pts2_r = pts2_r * scale2.unsqueeze(1)

pts1_np, pts2_np = pts1_r.cpu().numpy(), pts2_r.cpu().numpy()
conf_np = confidence.cpu().numpy()

M, inlier_mask = cv2.estimateAffinePartial2D(
pts1_np, pts2_np, method=cv2.RANSAC,
ransacReprojThreshold= model_config.feature_matching.pts_match_thresh,
maxIters=3000,
confidence=0.999
)

if M is not None and inlier_mask is not None:
    inlier_mask = inlier_mask.ravel().astype(bool)
    inliers_abs = int(inlier_mask.sum())
    outliers_abs = len(pts1) - inliers_abs
    inliers_p = inliers_abs / len(pts1) if len(pts1) > 0 else 0.0
    outliers_p = outliers_abs / len(pts1) if len(pts1) > 0 else 0.0
print(f'Points matching parameters: \n inliers: {inliers_abs} ({inliers_p}) \n outliers: {outliers_abs} ({outliers_p})')

Filtered
Matched points: 4341/10554.40625 (0.41129741429083233)
Confidence: mean: 0.49777787923812866, std: 0.21216867864131927, median: 0.4536406993865967
Points matching parameters: 
 inliers: 1509 (0.7541229385307346) 
 outliers: 492 (0.24587706146926536)
